# GameTheory-02c-Travelers-Dilemma

> **Navigation** : [Notebooks](../) | [Série GameTheory](README.md) | [<< 2-NormalForm (palier parent)](GameTheory-02-NormalForm.ipynb) | [2b - Définitions Lean](GameTheory-02b-Lean-Definitions.ipynb)

> **Série** : GameTheory-02c-Travelers-Dilemma — accrétion du palier 02 (forme normale, dominance, élimination itérée) : le Dilemme des Voyageurs de Basu (1994)


## Le paradigme : un accident de compagnie aérienne

Deux voyageurs reviennent d'une expédition. La compagnie a perdu les **deux bagages**, dont le contenu est **identique**. Le directeur de la compagnie leur demande de soumettre chacun, **indépendamment** et **sans communiquer**, une valeur de réclamation entre **2** et **100** dollars.

La règle d'indemnisation est astucieuse :

* Le directeur paie aux deux le **minimum** des deux réclamations.
* Celui qui a écrit le **plus petit** montant reçoit un **bonus** de **2 $** pour son honnêteté.
* Celui qui a écrit le **plus grand** montant subit une **pénalité** de **2 $** pour avoir exagéré.

> **Basu (1994), "The Traveler's Dilemma: Paradoxes of Rationality in Game Theory", *American Economic Review* 84(2), 391-395.**

### Pourquoi ce jeu est un paradigme

Ce jeu est l'un des objets les plus purs du **paradoxe de la rationalité** en théorie des jeux : il n'a ni répétition dans le temps, ni non-convexité cachée — seulement une forme normale à deux joueurs, stratégies bornées, gain continu par morceaux. Et pourtant, la raison formelle et l'intuition humaine divergent **radicalement** :

* La théorie prédit que **deux voyageurs rationnels réclament 2 $** (l'équilibre de Nash).
* Les joueurs humains, eux, réclament le plus souvent **100 $** (ou un montant proche).

Le but du notebook est de **montrer les deux côtés** : construire le raisonnement d'élimination qui mène à (2,2), puis regarder ce qui se passe quand on **varié le bonus $r$** — et voir que le paradoxe, lui, a un **seuil précis** en dessous duquel il disparaît.

### Source primaire

L'énoncé formel et la structure de paiement sont lus **firsthand** dans l'article de Basu (1994). La version standard (r=2, bornes 2..100) est celle reproduite ici. La forme générale de la fonction de paiement paramétrée par le bonus $r$ (et le plafond $\bar c$) est celle qui permet l'analyse de sensibilité du §4.

| Élément | Valeur |
|---|---|
| Bornes de réclamation | $c_i \in [2, 100]$ |
| Bonus d'honnêteté | $r = 2$ |
| Pénalité d'exagération | $r = 2$ |
| Lien | [JSTOR 2117865](https://www.jstor.org/stable/2117865) |

## 1. La fonction de paiement : l'objet formel

### La règle de l'arbitre, en une expression

Soient $x$ la réclamation du joueur Ligne et $y$ celle du joueur Colonne, $r$ le bonus/pénalité, $\bar c$ le plafond commun ($\bar c = 100$ ici). Le paiement de Ligne vaut :

$$
u_1(x, y) = \begin{cases}
   \min(x,y) + r & \text{si } x < y \\[2pt]
   \min(x,y) - r & \text{si } x > y \\[2pt]
   x & \text{si } x = y
\end{cases}
$$

La fonction est symétrique pour le joueur 2. On la code directement, avec `r` et `c_max` en paramètres — c'est ce qui permettra la sensibilité du §4.

### Import et fonction de paiement

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
import sys

# Portee de la sensibilite : bonus r variable, plafond fixe
C_MIN, C_MAX = 2, 100

def u1(x: int, y: int, r: float = 2.0, c_max: int = C_MAX) -> float:
    """Paiement du joueur Ligne. x et y dans [C_MIN, c_max]."""
    m = min(x, y)
    if x < y:
        return m + r
    if x > y:
        return m - r
    return m

def u2(x: int, y: int, r: float = 2.0, c_max: int = C_MAX) -> float:
    """Paiement du joueur Colonne (symetrie)."""
    return u1(y, x, r, c_max)

print("u1(90, 100, r=2) =", u1(90, 100, r=2), "   (90 < 100 -> min=90 + bonus 2)")
print("u1(100, 90, r=2) =", u1(100, 90, r=2), "   (100 > 90 -> min=90 - penalite 2)")
print("u1(90, 90, r=2)  =", u1(90, 90, r=2),  "   (egalite -> 90)")

u1(90, 100, r=2) = 92    (90 < 100 -> min=90 + bonus 2)
u1(100, 90, r=2) = 88    (100 > 90 -> min=90 - penalite 2)
u1(90, 90, r=2)  = 90    (egalite -> 90)


### Lecture du résultat

Trois cas se lisent comme attendu : réclamer **moins** que l'autre rapporte le minimum **majoré** du bonus ; réclamer **plus** le minimum **minoré** de la pénalité ; l'égalité donne le montant commun. La dissymétrie est le moteur de tout le paradoxe : baisser sa réclamation est toujours **au moins** aussi bon que la maintenir, dès lors que l'autre est en dessous.

## 2. La meilleure réponse : se tenir juste en dessous de l'autre

### Définition

La **meilleure réponse** de Ligne à une réclamation $y$ de Colonne est la valeur de $x$ qui maximise $u_1(x, y)$. On l'énumère directement sur la grille bornée.

In [2]:
def best_response(y: int, r: float = 2.0, c_max: int = C_MAX) -> tuple[int, float]:
    """Meilleure reponse de Ligne a la reclamation y de Colonne."""
    vals = [(x, u1(x, y, r, c_max)) for x in range(C_MIN, c_max + 1)]
    x_star, v_star = max(vals, key=lambda t: t[1])
    return x_star, v_star

print("y=100 -> x*", best_response(100))
print("y=2   -> x*", best_response(2))
print("y=55  -> x*", best_response(55))
print("y=3   -> x*", best_response(3))

y=100 -> x* (99, 101.0)
y=2   -> x* (2, 2)
y=55  -> x* (54, 56.0)
y=3   -> x* (2, 4.0)


### Lecture du résultat

La meilleure réponse est **systématiquement `max(C_MIN, y-1)`** : réclamer *un de moins* que l'autre donne le minimum (=$y-1$) **plus** le bonus, soit $y-1+2 = y+1$, ce qui bat réclamer la même chose ($y$) ou davantage. Seule exception : quand l'autre a déjà réclamé le **plancher** ($y=2$), on ne peut pas descendre en dessous — on est forcé à l'égalité (2,2).

## 3. L'élimination itérée des stratégies strictement dominées

### Le raisonnement qui mène à (2,2)

La meilleure réponse à une réclamation $y$ est **$y-1$** : réclamer un de moins que l'autre rapporte $y-1+r$ (le minimum majoré du bonus), ce qui bat réclamer la même chose ($y$) ou davantage. La conséquence n'est pas un gain global immédiat, mais une **élimination de haut en bas** : dans le jeu réduit aux stratégies restantes, la plus haute réclamation $k$ est strictement dominée par $k-1$ — les seules situations où $k-1$ était moins bon (un adversaire réclamant au-dessus de $k$) ont déjà été éliminées. En cascade, 100 est éliminé, puis 99, … jusqu'à ce qu'il ne reste que **$x = 2$**.

In [3]:
def iteratively_dominated(c_max: int = C_MAX) -> tuple[list[int], list[int]]:
    """Elimination iteree des strategies strictement dominees (de haut en bas).

    Dans le jeu reduit a l'ensemble `remaining`, la strategie x est strictement
    dominee par x-1 si x-1 rapporte au moins autant partout (sur `remaining`) ET
    strictement plus quelque part. A chaque etape la plus haute restante est
    dominee ; on l'elimine et on recommence jusqu'au plancher.
    """
    def dominated_by_prev(x, remaining, r=2.0):
        if x - 1 not in remaining:
            return False
        weakly = all(u1(x - 1, y, r, c_max) >= u1(x, y, r, c_max) for y in remaining)
        strictly = any(u1(x - 1, y, r, c_max) > u1(x, y, r, c_max) for y in remaining)
        return weakly and strictly

    remaining = list(range(C_MIN, c_max + 1))
    eliminated = []
    while True:
        candidates = [x for x in remaining
                      if x > C_MIN and dominated_by_prev(x, remaining)]
        if not candidates:
            break
        to_eliminate = max(candidates)  # on elimine la plus haute d'abord
        remaining.remove(to_eliminate)
        eliminated.append(to_eliminate)
    return remaining, eliminated

final, order = iteratively_dominated()
print("Strategies eliminées (du haut vers le bas), extrait:", order[:6], "...")
print("Strategies survivantes:", final, " (unique equilibre = plancher)")
print()
# Verification : a l'equilibre (2,2), aucun joueur ne veut bouger seul
print("u1(2,2) =", u1(2,2), " | devier a 3 pour Ligne :", u1(3,2), " -> pas profitable")
print("u1(2,2) =", u1(2,2), " | devier a 3 pour Colonne :", u2(2,3), " -> pas profitable")

Strategies eliminées (du haut vers le bas), extrait: [100, 99, 98, 97, 96, 95] ...
Strategies survivantes: [2]  (unique equilibre = plancher)

u1(2,2) = 2  | devier a 3 pour Ligne : 0.0  -> pas profitable
u1(2,2) = 2  | devier a 3 pour Colonne : 0.0  -> pas profitable


### Lecture du résultat

L'élimination itérée ne laisse qu'une seule stratégie : **réclamer 2 $**. À (2,2), chaque joueur reçoit 2 $, et s'écarter seul (monter) fait passer à $\min=2 - r$ : **0 $**, donc strictement pire. L'équilibre de Nash unique du jeu est donc **$(2,2)$** — la prédiction formelle. C'est l'apogée du paradoxe : la théorie dit « le minimum », l'intuition humaine dit autre chose (voir §4 et §5).

## 4. La sensibilité au bonus : où naît le paradoxe

### Le paradoxe dépend de $r$

Le raisonnement d'élimination ci-dessus ne tient pas pour tout $r$ : il exige **$r \ge 1$**. Comparer la plus haute réclamation $k$ à $k-1$ face à un adversaire qui réclame lui aussi $k$ donne $k-1+r$ contre $k$ — le sous-cotage ne rapporte qu'à partir de $r = 1$. La table du §4 ci-dessous le mesure : $r = 0{,}5$ n'atteint pas le plancher, $r = 1$ si. Mais l'expérience — humaine et expérimentale — montre que les gens jouent haut **quand $r$ est petit** et bas **quand $r$ est grand**. Le paradoxe n'est pas un fait : c'est une fonction de $r$.

On mesure la différence entre l'équilibre théorique (2) et la prédiction « naïve » qu'un joueur stratégiquement **superficiel** ferait : viser le maximum $\bar c$, puisqu'il s'agit avant tout de capter le bonus.

In [4]:
def undercut_gain(c: int, r: float = 2.0) -> float:
    """Gain a devier de c vers c-1 quand l'autre tient c : u1(c-1,c) - u1(c,c).

    Independe de c : vaut (c-1+r) - c = r - 1. C'est le moteur de la spirale.
    """
    return u1(c - 1, c, r) - u1(c, c, r)

def elimination_reaches_floor(r: float = 2.0, c_max: int = C_MAX) -> bool:
    """L'elimination de haut en bas atteint-elle le plancher pour ce bonus r ?

    Reutilise le raisonnement de la cellule 3 (domination stricte de la plus
    haute restante par son precedesseur), en propageant r partout.
    """
    def dominated_by_prev(x, remaining):
        if x - 1 not in remaining:
            return False
        weakly = all(u1(x - 1, y, r, c_max) >= u1(x, y, r, c_max) for y in remaining)
        strictly = any(u1(x - 1, y, r, c_max) > u1(x, y, r, c_max) for y in remaining)
        return weakly and strictly

    remaining = list(range(C_MIN, c_max + 1))
    while True:
        candidates = [x for x in remaining
                      if x > C_MIN and dominated_by_prev(x, remaining)]
        if not candidates:
            break
        remaining.remove(max(candidates))
    return remaining == [C_MIN]

# Le moteur du paradoxe : le gain de sous-cotage r-1, et le seuil r*=1.
rs = [0.5, 1.0, 1.5, 2.0, 5.0, 10.0]
print("  r  |  gain de sous-cotage (r-1)  |  la spirale atteint (2,2) ?")
for r in rs:
    g = undercut_gain(50, r)
    ok = elimination_reaches_floor(r)
    print(f"{r:>4} | {g:>8} | {'OUI' if ok else 'NON'}")

print()
print("Lecture : le gain de sous-cotage vaut r-1, independant du point de depart c.")
print("  -> r < 1 : sous-coter ne rapporte rien (gain negatif) -> grimper n'est pas puni,")
print("             la spirale ne demarre pas -> le paradoxe est VIVANT (humains jouent haut).")
print("  -> r > 1 : sous-coter devient rentable -> la spirale descend forcement vers (2,2)")
print("             -> le paradoxe SE DISSOUT. Le seuil est r* = 1.")

  r  |  gain de sous-cotage (r-1)  |  la spirale atteint (2,2) ?
 0.5 |     -0.5 | NON
 1.0 |      0.0 | OUI


 1.5 |      0.5 | OUI


 2.0 |      1.0 | OUI


 5.0 |      4.0 | OUI


10.0 |      9.0 | OUI

Lecture : le gain de sous-cotage vaut r-1, independant du point de depart c.
  -> r < 1 : sous-coter ne rapporte rien (gain negatif) -> grimper n'est pas puni,
             la spirale ne demarre pas -> le paradoxe est VIVANT (humains jouent haut).
  -> r > 1 : sous-coter devient rentable -> la spirale descend forcement vers (2,2)
             -> le paradoxe SE DISSOUT. Le seuil est r* = 1.


### Lecture du résultat

Le moteur du paradoxe est le **gain de sous-cotage** : si l'autre réclame $c$, réclamer $c-1$ rapporte $(c-1)+r$ au lieu de $c$ (l'égalité), soit un gain de $r-1$ — **indépendant** du point de départ. Ce gain unique pilote toute la spirale.

* **$r < 1$** : le gain de sous-cotage est négatif — sous-coter ne rapporte rien, grimper n'est pas puni. L'élimination de haut en bas **ne démarre même pas** : un joueur peut rationnellement rester haut, et les humains jouent en effet ~100 $. **Le paradoxe est vivant.**
* **$r > 1$** : le gain est positif — sous-coter devient rentable à chaque étage, la spirale descend **forcément** jusqu'à (2,2), et la prédiction théorique coïncide avec le comportement. **Le paradoxe se dissout.**

Le seuil est donc **$r^* = 1$** (dans la version canonique $r=2$, on est bien au-dessus). C'est exactement le fait expérimental de Basu : la gravité du paradoxe est une fonction de $r$, pas une constante.

## 5. Le contre-claim : l'équilibre de l'EEI est réel, mais il n'est pas la prédiction comportementale


### La question que l'article ne tait pas

L'énoncé exact — « l'équilibre de Nash est (2,2) » — est **vrai**, mais il n'implique **pas** que des joueurs humains le choisissent. Le contre-claim ici n'est pas de contredire le théorème, mais de **délimiter sa portée** : ce que l'élimination itérée *prouve* (un équilibre unique) est distinct de ce qu'elle *prédit* (le comportement). L'écart est exhibé quand on relâche l'hypothèse de rationalité parfaite et de connaissance commune.

On le montre en simulant un **jeu où un joueur n'est pas certain** que l'autre descend au plancher : s'il existe une probabilité $\varepsilon$ que l'autre « grimpe », le meilleur-pari pour le joueur prudent devient de **grimper** — et l'équilibre (2,2) cesse d'être un point stable de la dynamique d'adaptation.

In [5]:
def expected_u1_given_mix(my_x: int, opponent_dist, r: float = 2.0, c_max: int = C_MAX) -> float:
    """Esperance de paiement si mon adversaire tire y ~ opponent_dist."""
    return sum(p * u1(my_x, y, r, c_max) for y, p in opponent_dist)

def aggregate_BE(opponent_dist, r: float = 2.0, c_max: int = C_MAX) -> int:
    """Meilleure reponse integrale face a une distribution de l'autre."""
    return max(range(C_MIN, c_max + 1), key=lambda x: expected_u1_given_mix(x, opponent_dist, r, c_max))

# Hypotheses : une fraction eps de l'autre 'grimpe' (uniforme sur [c_max//2, c_max]),
# le reste joue l'equilibre (2).
for eps in [0.0, 0.05, 0.2, 0.5]:
    dist = []
    dist.append((C_MIN, 1 - eps))
    for y in range(C_MAX // 2, C_MAX + 1):
        dist.append((y, eps / (C_MAX // 2 + 1)))
    x_be = aggregate_BE(dist)
    print(f"eps={eps:>4.2f} :: meilleure reponse de Ligne = {x_be:>3}")

print()
print("Lecture : des qu'une fraction non nulle de l'autre peut grimper, la meilleure")
print("reponse saute du plancher (2) vers le haut. L'equilibre (2,2) exige la confiance")
print("ABSOLUE dans la rationalite de l'autre — c'est sa condition de stabilite.")

eps=0.00 :: meilleure reponse de Ligne =   2
eps=0.05 :: meilleure reponse de Ligne =  96
eps=0.20 :: meilleure reponse de Ligne =  96
eps=0.50 :: meilleure reponse de Ligne =  96

Lecture : des qu'une fraction non nulle de l'autre peut grimper, la meilleure
reponse saute du plancher (2) vers le haut. L'equilibre (2,2) exige la confiance
ABSOLUE dans la rationalite de l'autre — c'est sa condition de stabilite.


### Verdict du contre-claim

Le contre-claim n'est pas « le théorème est faux » — il est **vrai**. Le contre-claim est : **l'équilibre de l'élimination itérée n'est pas une prédiction comportementale**. Il ne tient que sous connaissance commune de la rationalité. Dès qu'une probabilité $\varepsilon>0$ de « grimper » entre, la meilleure réponse du joueur prudent saute vers le haut (la simulation ci-dessus l'exhibe) — donc (2,2) est un équilibre *fragile*, pas un attracteur. C'est exactement la leçon que le notebook de distillation devait faire émerger, et qui nourrit l'expérience ICT candidate (voir §6).

## 6. Exercices (3) — à compléter par l'étudiant

Les trois exercices ci-dessous sont des stubs **C.1** (ils s'exécutent sans erreur, mais laissent le raisonnement à l'étudiant). Contenu réel à produire, jamais une valeur fabriquée.


In [6]:
# Exercice 1 : le jeu est-il antagoniste, ou peut-il devenir un jeu de coordination ?
# Question : existe-t-il une reclamation COMMUNE x > 2 qui rapporte strictement plus
#            que le plancher (2,2) aux DEUX joueurs a la fois ? Le verdict change-t-il
#            avec r ?
# Indice : sur la diagonale x = y, bonus et penalite s'annulent.

def coordination_possible(r: float = 2.0, c_max: int = C_MAX):
    """Existe-t-il x dans ]C_MIN, c_max] tel que u1(x, x, r) > u1(C_MIN, C_MIN, r) ?

    Renvoyer True ou False. Tant que l'exercice n'est pas traite, la fonction
    renvoie None et l'appel ci-dessous l'annonce sans lever d'erreur (regle C.1).
    """
    # TODO etudiant : parcourir x de C_MIN + 1 a c_max, comparer u1(x, x, r) au
    #                 paiement du plancher u1(C_MIN, C_MIN, r), renvoyer le verdict.
    return None

for r_test in [0.5, 2.0, 25.0]:
    verdict = coordination_possible(r_test)
    print(f"r={r_test:>5.1f} :: coordination_possible ->",
          "Exercice a completer" if verdict is None else verdict)

r=  0.5 :: coordination_possible -> Exercice a completer
r=  2.0 :: coordination_possible -> Exercice a completer
r= 25.0 :: coordination_possible -> Exercice a completer


### Exercice 2 — le seuil de dissolution du paradoxe

Cherchez la valeur de $r$ au-delà de laquelle « jouer le plafond » (l'égalité au maximum) rapporte **strictement plus** que « jouer le plancher » — comparez $u_1(\bar c,\bar c)$ et $u_1(2,2)$, puis dites si le verdict dépend de $r$. Justifiez à partir de la définition de $u_1$ sur la diagonale $x = y$.


In [7]:
# Exercice 2 : le plafond bat-il le plancher, et cela depend-il de r ?
# Question : l'egalite au plafond (c_max, c_max) rapporte-t-elle strictement plus que
#            l'egalite au plancher (2, 2) ? Ecrivez la comparaison, puis expliquez en
#            une phrase pourquoi r n'apparait pas dans le resultat.

def plafond_bat_plancher(r: float = 2.0, c_max: int = C_MAX):
    """u1(c_max, c_max, r) > u1(C_MIN, C_MIN, r) ? (None tant que non traite)"""
    # TODO etudiant : ecrire la comparaison des deux paiements diagonaux.
    return None

for r_test in [0.5, 2.0, 10.0, 50.0]:
    verdict = plafond_bat_plancher(r_test)
    print(f"r={r_test:>5.1f} :: plafond_bat_plancher ->",
          "Exercice a completer" if verdict is None else verdict)

r=  0.5 :: plafond_bat_plancher -> Exercice a completer
r=  2.0 :: plafond_bat_plancher -> Exercice a completer
r= 10.0 :: plafond_bat_plancher -> Exercice a completer
r= 50.0 :: plafond_bat_plancher -> Exercice a completer


### Exercice 3 — le coût de l'optimisme

Reprenez le calcul de l'espérance du §5 et trouvez la **valeur de $\varepsilon$** à partir de laquelle la meilleure réponse d'un joueur prudent passe du plancher (2) à un montant **strictement supérieur à 2**. C'est le « seuil de bascule » de la confiance.


In [8]:
# Exercice 3 : le seuil de bascule de la confiance
# Objectif : trouver la plus petite probabilite eps de "grimper" chez l'adversaire pour
#            laquelle la meilleure reponse d'un joueur prudent QUITTE le plancher C_MIN.
# Indice : reutilisez aggregate_BE du paragraphe 5 et la distribution qui y est
#          construite ((C_MIN, 1 - eps), puis une queue uniforme sur [c_max // 2, c_max]).

def seuil_bascule(c_max: int = C_MAX, r: float = 2.0):
    """Plus petit eps tel que aggregate_BE(dist(eps), r, c_max) > C_MIN.

    Renvoyer un flottant, ou None si aucun eps du balayage ne fait basculer la
    meilleure reponse. Tant que l'exercice n'est pas traite, renvoie None (C.1).
    """
    # TODO etudiant : balayer eps par pas fin sur [0, 1[, construire la distribution
    #                 du paragraphe 5, renvoyer le premier eps qui fait sauter la
    #                 meilleure reponse au-dessus du plancher.
    return None

s = seuil_bascule()
if s is None:
    print("Exercice a completer : seuil de bascule eps* non determine.")
else:
    print("Seuil de bascule eps* =", s)
    print("Lecture attendue : au-dela de eps*, la meilleure reponse quitte le plancher —")
    print("(2,2) cesse donc d'etre la meilleure reponse d'un joueur prudent.")

Exercice a completer : seuil de bascule eps* non determine.


### À retenir

* **L'objet formel** : un jeu à deux joueurs à stratégies bornées $[2,\bar c]$, paiement $\min(x,y)$ plus bonus $r$ au plus petit / pénalité $r$ au plus grand.
* **Le claim exact** : l'élimination itérée des stratégies strictement dominées laisse l'unique équilibre $(2,2)$.
* **Le contre-claim** : cet équilibre n'est pas une prédiction comportementale — il ne tient que sous connaissance commune de la rationalité. Une probabilité $\varepsilon>0$ de jouer haut fait sauter la meilleure réponse vers le plafond.
* **Le seuil** : le paradoxe se dissout quand $r$ devient grand ; il n'est paradoxal que pour $r$ petit.

> **Expérience ICT candidate** : mesurer en TP le taux de « montée » (réclamation > 50) pour $r=2$ vs $r=25$, et confronter à la courbe bascule de l'exercice 3 — c'est un test falsifiable du contre-claim.

> **Palier parent** : [GameTheory-02-NormalForm](GameTheory-02-NormalForm.ipynb) (forme normale, dominance, élimination itérée) · **Sibling** : [GameTheory-02b-Lean-Definitions](GameTheory-02b-Lean-Definitions.ipynb) · **Série** : [GameTheory](README.md)